In [64]:
import numpy as np
from tensorflow.keras.datasets import mnist

# 데이터 불러오기
(x_train, t_train), (x_test, t_test) = mnist.load_data()

# 이미지 정규화
x_train = x_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0

# 28x28 이미지를 784차원으로 변환
x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

# One-hot encoding
t_train = np.eye(10)[t_train]
t_test = np.eye(10)[t_test]

In [65]:
import numpy as np

class Affine:
    def __init__(self, W, b):
        self.W = W
        self.b = b
        self.x = None
        self.dW = None
        self.db = None

    def forward(self, x):
        self.x = x
        out = np.dot(x, self.W) + self.b

        return out

    def backward(self, dout):
        dx = np.dot(dout, self.W.T)
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis=0)

        return dx

# 입력 데이터
x = np.array([[1, 2],
              [3, 4],
              [5, 6]])

# 가중치와 편향
W = np.array([[0.1, 0.2, 0.3],
              [0.4, 0.5, 0.6]])

b = np.array([0.1, 0.2, 0.3])

# Affine 계층 생성
affine = Affine(W, b)

# 순방향 계산
out = affine.forward(x)
print(out)

# 역방향으로 전달할 기울기
dout = np.array([[1, 2, 3],
                 [4, 5, 6],
                 [7, 8, 9]])

# 역방향 계산
dx = affine.backward(dout)

print("dx:")
print(dx)

print("dW:")
print(affine.dW)

print("db:")
print(affine.db)

[[1.  1.4 1.8]
 [2.  2.8 3.6]
 [3.  4.2 5.4]]
dx:
[[ 1.4  3.2]
 [ 3.2  7.7]
 [ 5.  12.2]]
dW:
[[48 57 66]
 [60 72 84]]
db:
[12 15 18]


In [66]:
def softmax(a):
    c = np.max(a)
    exp_a = np.exp(a - c)
    y = exp_a / np.sum(exp_a)
    return y

def cross_entropy_error(y_pred, y_true):
  # log(0) 에러 방지를 위해 아주 작은 값(epsilon)을 더함
  delta = 1e-7
  batch_size = y_pred.shape[0]

  # 원-핫 인코딩된 타겟일 경우
  loss = -np.sum(y_true * np.log(y_pred + delta)) / batch_size
  return loss



class SoftmaxWithLoss:
    def __init__(self):
        self.loss = None
        self.y = None
        self.t = None

    def forward(self, x, t):
        self.t = t
        self.y = softmax(x)
        self.loss = cross_entropy_error(self.y, self.t)

        return self.loss

    def backward(self, dout=1):
        batch_size = self.t.shape[0]
        dx = (self.y - self.t) / batch_size

        return dx

# 입력 데이터 (3개 클래스에 대한 점수)
x = np.array([0.3, 2.9, 4.0])

# 정답 데이터 (클래스 2가 정답)
t = np.array([0, 0, 1])

# SoftmaxWithLoss 계층 생성
layer = SoftmaxWithLoss()

# 순방향 계산
loss = layer.forward(x, t)

print("Softmax output:")
print(layer.y)

print("\nLoss:")
print(loss)

# 역방향 계산
dx = layer.backward()

print("\nBackward output:")
print(dx)

Softmax output:
[0.01821127 0.24519181 0.73659691]

Loss:
0.10190477635100009

Backward output:
[ 0.00607042  0.0817306  -0.08780103]


In [67]:
import sys
import os
import numpy as np
from collections import OrderedDict
sys.path.append('/content/sample_data')
from layers import *
from gradient import numerical_gradient

class TwoLayerNet:
    def __init__(self, input_size, hidden_size, output_size,
        weight_init_std=0.01):
        # 가중치 초기화
        self.params = {}
        self.params['W1'] = weight_init_std * \
            np.random.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std * \
            np.random.randn(hidden_size, output_size)
        self.params['b2'] = np.zeros(output_size)

        # 계층 생성
        self.layers = OrderedDict()
        self.layers['Affine1'] = \
            Affine(self.params['W1'], self.params['b1'])
        self.layers['Relu1'] = Relu()
        self.layers['Affine2'] = \
            Affine(self.params['W2'], self.params['b2'])
        self.lastLayer = SoftmaxWithLoss()

    def predict(self, x):
        for layer in self.layers.values():
            x = layer.forward(x)

        return x

    # x : 입력 데이터, t : 정답 레이블
    def loss(self, x, t):
        y = self.predict(x)
        return self.lastLayer.forward(y, t)

    def accuracy(self, x, t):
        y = self.predict(x)
        y = np.argmax(y, axis=1)
        if t.ndim != 1:
            t = np.argmax(t, axis=1)

        accuracy = np.sum(y == t) / float(x.shape[0])
        return accuracy

    def numerical_gradient(self, x, t):
        loss_W = lambda W: self.loss(x, t)

        grads = {}
        grads['W1'] = numerical_gradient(loss_W, self.params['W1'])
        grads['b1'] = numerical_gradient(loss_W, self.params['b1'])
        grads['W2'] = numerical_gradient(loss_W, self.params['W2'])
        grads['b2'] = numerical_gradient(loss_W, self.params['b2'])

        return grads

    def gradient(self, x, t):
        # 순전파
        self.loss(x, t)

        # 역전파
        dout = 1
        dout = self.lastLayer.backward(dout)

        layers = list(self.layers.values())
        layers.reverse()
        for layer in layers:
            dout = layer.backward(dout)

        # 결과 저장
        grads = {}
        grads['W1'] = self.layers['Affine1'].dW
        grads['b1'] = self.layers['Affine1'].db
        grads['W2'] = self.layers['Affine2'].dW
        grads['b2'] = self.layers['Affine2'].db

        return grads


In [68]:
network = TwoLayerNet(input_size=784, hidden_size=50, output_size=10)

x_batch = x_train[:3]
t_batch = t_train[:3]

grad_numerical = network.numerical_gradient(x_batch, t_batch)
grad_backprop = network.gradient(x_batch, t_batch)

    # 각 가중치의 차이의 절댓값을 구한 후, 그 절댓값들의 평균을 낸다.
for key in grad_numerical.keys():
        diff = np.average(np.abs(grad_backprop[key] - grad_numerical[key]))
        print(key + ":" + str(diff))

W1:2.294741955260151e-13
b1:8.229158034254308e-13
W2:8.133328087628691e-13
b2:1.1968204899348578e-10


In [70]:
import sys
import os
import numpy as np
sys.path.append('/content/sample_data')




# 하이퍼 파라메터
iters_num = 10000  # 반복횟수
train_size = x_train.shape[0]
batch_size = 100  # 미니배치 크기
learning_rate = 0.1

train_loss_list = []
train_acc_list = []
test_acc_list = []

# 1에폭당 반복 수
iter_per_epoch = max(train_size / batch_size, 1)

for i in range(iters_num):
    # print(i)
    # 미니배치 획득
    batch_mask = np.random.choice(train_size, batch_size)
    x_batch = x_train[batch_mask]
    t_batch = t_train[batch_mask]

    # 오차역전파법으로 기울기 계산
    grad = network.gradient(x_batch, t_batch)

    # 매개변수 갱신
    for key in ('W1', 'b1', 'W2', 'b2'):
        network.params[key] -= learning_rate * grad[key]

    # 학습 경과 기록
    loss = network.loss(x_batch, t_batch)
    train_loss_list.append(loss)

    # 1에폭 당 정확도 계산
    if i % iter_per_epoch == 0:
        train_acc = network.accuracy(x_train, t_train)
        test_acc = network.accuracy(x_test, t_test)
        train_acc_list.append(train_acc)
        test_acc_list.append(test_acc)
        print("train acc, test acc | " + str(train_acc) + ", " + str(test_acc))

train acc, test acc | 0.9735, 0.9661
train acc, test acc | 0.9763166666666667, 0.968
train acc, test acc | 0.9772166666666666, 0.9683
train acc, test acc | 0.9787833333333333, 0.9698
train acc, test acc | 0.9800666666666666, 0.9709
train acc, test acc | 0.9804, 0.9698
train acc, test acc | 0.9812333333333333, 0.9712
train acc, test acc | 0.9828333333333333, 0.9705
train acc, test acc | 0.9833666666666666, 0.9706
train acc, test acc | 0.9847833333333333, 0.973
train acc, test acc | 0.9848, 0.9732
train acc, test acc | 0.9847, 0.9717


KeyboardInterrupt: 